In [131]:
#import
import pandas as pd
from sklearn.model_selection import train_test_split  #データの分割

from sklearn.preprocessing import StandardScaler #標準化
from sklearn.model_selection import KFold
from sklearn.model_selection import cross_validate

from sklearn.linear_model import LinearRegression #回帰
from sklearn.preprocessing import PolynomialFeatures  #交互作用特徴量
from sklearn.linear_model import Ridge  #リッジ回帰
from sklearn.linear_model import Lasso  #ラッソ回帰

In [132]:
df = pd.read_csv('datafiles/train.csv')

In [133]:
#欠損値の確認
for c in df.columns:
    null_counts = df[c].isnull().sum()
    if null_counts != 0:
        print(f'{null_counts}  列＝{c}')

259  列＝LotFrontage
1369  列＝Alley
872  列＝MasVnrType
8  列＝MasVnrArea
37  列＝BsmtQual
37  列＝BsmtCond
38  列＝BsmtExposure
37  列＝BsmtFinType1
38  列＝BsmtFinType2
1  列＝Electrical
690  列＝FireplaceQu
81  列＝GarageType
81  列＝GarageYrBlt
81  列＝GarageFinish
81  列＝GarageQual
81  列＝GarageCond
1453  列＝PoolQC
1179  列＝Fence
1406  列＝MiscFeature


In [134]:
#明らかに不要な'id'を除く
df = df.drop(['Id'], axis = 1)

In [135]:
#ダミー変数化する行の抜き出し
to_dummy_cols = []
for c in df.columns:
    if type(c) == str:
        to_dummy_cols.append(c)
print(to_dummy_cols)

['MSSubClass', 'MSZoning', 'LotFrontage', 'LotArea', 'Street', 'Alley', 'LotShape', 'LandContour', 'Utilities', 'LotConfig', 'LandSlope', 'Neighborhood', 'Condition1', 'Condition2', 'BldgType', 'HouseStyle', 'OverallQual', 'OverallCond', 'YearBuilt', 'YearRemodAdd', 'RoofStyle', 'RoofMatl', 'Exterior1st', 'Exterior2nd', 'MasVnrType', 'MasVnrArea', 'ExterQual', 'ExterCond', 'Foundation', 'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinSF1', 'BsmtFinType2', 'BsmtFinSF2', 'BsmtUnfSF', 'TotalBsmtSF', 'Heating', 'HeatingQC', 'CentralAir', 'Electrical', '1stFlrSF', '2ndFlrSF', 'LowQualFinSF', 'GrLivArea', 'BsmtFullBath', 'BsmtHalfBath', 'FullBath', 'HalfBath', 'BedroomAbvGr', 'KitchenAbvGr', 'KitchenQual', 'TotRmsAbvGrd', 'Functional', 'Fireplaces', 'FireplaceQu', 'GarageType', 'GarageYrBlt', 'GarageFinish', 'GarageCars', 'GarageArea', 'GarageQual', 'GarageCond', 'PavedDrive', 'WoodDeckSF', 'OpenPorchSF', 'EnclosedPorch', '3SsnPorch', 'ScreenPorch', 'PoolArea', 'PoolQC', 'Fen

In [136]:
'''
#strの特徴量の中にNAが混ざっている列
Alley
MasVnrType
BsmtQual
BsmtCond
BsmtExposure
BsmtFinType1
BsmtFinType2
Electrical
FireplaceQu
GarageType
GarageFinish
GarageQual
GarageCond
PoolQC
Fence
MiscFeature

#intの特徴量の中にNAが混ざっている列
LotFrontage  NAを0に変更
MasVnrArea   NAを0に変更
GarageYrBlt  NAを0に変更
'''
#data_description.txt を確認すると、すべての特徴量で'NA'に意味があるようだったので補完
to_NA_cols = ['Alley', 'MasVnrType', 'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 
    'BsmtFinType2', 'Electrical', 'FireplaceQu', 'GarageType', 'GarageFinish', 'GarageQual', 
    'GarageCond', 'Fence', 'MiscFeature'
]
df[to_NA_cols] = df[to_NA_cols].fillna('NA')
df[['LotFrontage', 'MasVnrArea', 'GarageYrBlt']] = df[['LotFrontage', 'MasVnrArea', 'GarageYrBlt']].fillna(0.0)


In [137]:
print(df.columns)

Index(['MSSubClass', 'MSZoning', 'LotFrontage', 'LotArea', 'Street', 'Alley',
       'LotShape', 'LandContour', 'Utilities', 'LotConfig', 'LandSlope',
       'Neighborhood', 'Condition1', 'Condition2', 'BldgType', 'HouseStyle',
       'OverallQual', 'OverallCond', 'YearBuilt', 'YearRemodAdd', 'RoofStyle',
       'RoofMatl', 'Exterior1st', 'Exterior2nd', 'MasVnrType', 'MasVnrArea',
       'ExterQual', 'ExterCond', 'Foundation', 'BsmtQual', 'BsmtCond',
       'BsmtExposure', 'BsmtFinType1', 'BsmtFinSF1', 'BsmtFinType2',
       'BsmtFinSF2', 'BsmtUnfSF', 'TotalBsmtSF', 'Heating', 'HeatingQC',
       'CentralAir', 'Electrical', '1stFlrSF', '2ndFlrSF', 'LowQualFinSF',
       'GrLivArea', 'BsmtFullBath', 'BsmtHalfBath', 'FullBath', 'HalfBath',
       'BedroomAbvGr', 'KitchenAbvGr', 'KitchenQual', 'TotRmsAbvGrd',
       'Functional', 'Fireplaces', 'FireplaceQu', 'GarageType', 'GarageYrBlt',
       'GarageFinish', 'GarageCars', 'GarageArea', 'GarageQual', 'GarageCond',
       'PavedDrive', 'Wo

In [138]:
#float型に変更
not_to_dummy = ['MSSubClass', 'LotFrontage', 
    'LotArea', 'OverallQual', 'OverallCond', 'YearBuilt', 'YearRemodAdd', 
    'MasVnrArea', 'BsmtFinSF1', 'BsmtFinSF2', 'BsmtUnfSF', 'TotalBsmtSF', 
    '1stFlrSF', '2ndFlrSF', 'LowQualFinSF', 'GrLivArea', 'BsmtFullBath', 
    'BsmtHalfBath', 'FullBath', 'HalfBath', 'BedroomAbvGr', 'KitchenAbvGr', 
    'TotRmsAbvGrd', 'Fireplaces', 'GarageYrBlt', 'GarageCars', 'GarageArea', 
    'WoodDeckSF', 'OpenPorchSF', 'EnclosedPorch', '3SsnPorch',
    'ScreenPorch', 'PoolArea', 'MiscVal', 'MoSold', 'YrSold', 'SalePrice'
]

In [139]:
df[not_to_dummy] = df[not_to_dummy].astype('float64')

In [140]:
#ダミー変数化
to_dummy = set(df.columns) - set(not_to_dummy)
to_dummy = list(to_dummy)
for c in to_dummy:
    dummy = pd.get_dummies(df[c], prefix=c, drop_first = True, dtype = int)
    df = pd.concat([df, dummy], axis = 1)
    df = df.drop([c], axis = 1)

In [141]:
#float型に変更
df = df.astype('float64')

In [142]:
#説明変数と目的変数のデータフレームを作る
df_y = pd.DataFrame(df['SalePrice'])
df_x = df.drop(['SalePrice'], axis = 1)

#標準化
sc_model=StandardScaler()
sc_model.fit(df_x)
sc_x = sc_model.fit_transform(df_x)

In [143]:
#重回帰、リッジ回帰、ラッソ回帰、回帰木を実践し、結果を比較する。
kf=KFold(n_splits=5, shuffle=True, random_state=0)

In [144]:
#重回帰
model1 = LinearRegression()
result = cross_validate(model1, sc_x, df_y, cv = kf, scoring = 'r2', return_train_score = True)
print(sum(result['test_score'])/len(result['test_score']))

0.5498616628567776


In [145]:
#リッジ回帰
#正則化項の定数を0.01~20まで検証
best_ridgescore = 0
best_alpha = 0

#alpha（Fの係数）を1~100まで変化させて実験
for i in range(1,101):
    num = i
    ridgeModel = Ridge(random_state = 0,alpha = num)
    all_result = cross_validate(ridgeModel, sc_x, df_y, cv = kf, scoring = 'r2', return_train_score = True)
    result = sum(all_result['test_score'])/len(all_result['test_score'])
    if result > best_ridgescore:
        best_ridgescore = result
        best_alpha = num
print(f'正則化項＝{best_alpha}　リッジ回帰のスコア＝{best_ridgescore}')

#完成したリッジ回帰モデルで学習
model2 = Ridge(alpha = (best_alpha/100))
result = cross_validate(model2, sc_x, df_y, cv = kf, scoring = 'r2', return_train_score = True)
print(f'完成したモデルのスコア＝{sum(result['test_score'])/len(result['test_score'])}')

正則化項＝100　リッジ回帰のスコア＝0.7773772831801896
完成したモデルのスコア＝0.5838636191087039


In [146]:
#ラッソ回帰
#正則化項の定数を0.01~20まで検証
best_lassoscore = 0
best_alpha = 0
#alpha（Fの係数）を1~100まで変化させて実験
for i in range(1,101):
    num = i
    lassoModel = Lasso(random_state = 0, alpha = num)
    all_result = cross_validate(lassoModel, sc_x, df_y, cv = kf, scoring = 'r2', return_train_score = True)
    result = sum(all_result['test_score'])/len(all_result['test_score'])
    if result > best_lassoscore:
        best_lassoscore = result
        best_alpha = num
print(f'正則化項＝{best_alpha}　ラッソ回帰のスコア＝{best_lassoscore}')

#完成したラッソ回帰モデルで学習
model3 = Lasso(alpha = (best_alpha/100))
result = cross_validate(model3, sc_x, df_y, cv = kf, scoring = 'r2', return_train_score = True)
print(f'完成したモデルのスコア＝{sum(result['test_score'])/len(result['test_score'])}')


c:\Users\natsu\anaconda3\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.497e+11, tolerance: 7.191e+08
  model = cd_fast.enet_coordinate_descent(
c:\Users\natsu\anaconda3\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.959e+11, tolerance: 7.582e+08
  model = cd_fast.enet_coordinate_descent(
c:\Users\natsu\anaconda3\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.465e+11, toleranc

正則化項＝100　ラッソ回帰のスコア＝0.6355670754771634


c:\Users\natsu\anaconda3\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.497e+11, tolerance: 7.191e+08
  model = cd_fast.enet_coordinate_descent(
c:\Users\natsu\anaconda3\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.959e+11, tolerance: 7.582e+08
  model = cd_fast.enet_coordinate_descent(
c:\Users\natsu\anaconda3\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.465e+11, toleranc

完成したモデルのスコア＝0.5444905254885237


c:\Users\natsu\anaconda3\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 2.090e+11, tolerance: 7.732e+08
  model = cd_fast.enet_coordinate_descent(


In [ ]:
'''
単純なモデルでは、下記のような結果となった。

重回帰のスコア  0.5498616628567776
リッジ回帰のスコア  0.5838636191087039
ラッソ回帰のスコア  0.5444905254885237

以下では、特徴量を改善したモデルでも実験する。
'''

'\n単純なモデルでは、下記のような結果となった。\n\n重回帰のスコア  0.5498616628567776\nリッジ回帰のスコア  0.5838636191087039\nラッソ回帰のスコア  0.5444905254885237\n\n以下では、特徴量を改善したモデルでも実験する。\n'

In [148]:
model2.fit(sc_x, df_y)

,alpha,1.0
,fit_intercept,True
,copy_X,True
,max_iter,None
,tol,0.0001
,solver,'auto'
,positive,False
,random_state,None


In [149]:
#リッジ回帰の係数と切片の確認
coef_df = pd.DataFrame({
    'col': df_x.columns,
    'coef': model2.coef_
})
print(f'係数: {coef_df }')
print(f'切片: {model2.intercept_}')

係数:                        col          coef
0               MSSubClass  -2377.792247
1              LotFrontage    -80.836687
2                  LotArea   7160.997070
3              OverallQual   9518.322915
4              OverallCond   6492.241559
5                YearBuilt   9314.556753
6             YearRemodAdd   1918.465949
7               MasVnrArea   3812.766276
8               BsmtFinSF1   7414.371608
9               BsmtFinSF2   1702.931304
10               BsmtUnfSF   -118.872978
11             TotalBsmtSF   8214.785947
12                1stFlrSF   5738.540798
13                2ndFlrSF  13793.832629
14            LowQualFinSF  -1656.750437
15               GrLivArea  15527.303464
16            BsmtFullBath    847.278209
17            BsmtHalfBath   -158.446945
18                FullBath   2198.324367
19                HalfBath    967.921313
20            BedroomAbvGr  -2842.928556
21            KitchenAbvGr  -3042.257613
22            TotRmsAbvGrd   3258.356481
23          

In [ ]:
pd.set_option('display.max_columns', None) 
# 行を省略せずに表示
pd.set_option('display.max_rows', None)
coef_df.sort_values('coef', ascending=False)

,col,coef
62,RoofMatl_CompShg,76750.365033
66,RoofMatl_Tar&Grv,50206.039237
68,RoofMatl_WdShngl,40998.284633
67,RoofMatl_WdShake,33285.794558
73,GarageCond_TA,21206.538556
63,RoofMatl_Membran,17455.920278
64,RoofMatl_Metal,16648.146237
15,GrLivArea,15527.303464
65,RoofMatl_Roll,14891.960650
13,2ndFlrSF,13793.832629
